In [1]:
from datasets import load_from_disk
import datasets 
from transformers import AutoTokenizer 
import sys
sys.path.append("/mimer/NOBACKUP/groups/naiss2024-22-903/LLMedu")
from data.CyberMetric import CyberMetric_evaluator
from openai import OpenAI 
import re
import pandas as pd 

df = pd.read_excel('data/class1_8.ods', engine='odf')

/mimer/NOBACKUP/groups/naiss2024-22-903/anaconda3/envs/RL/lib/python3.8/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
import numpy as np
class read_topic(): 
    def __init__(self, df): 
        ''' 
        Input:  dataframe of .ods CSEC2017 file, with ID, Knowledge_units, Topics, Description as headers
                & index  
        Output: topic, description, and CSEC2017 label of provided index or random index if index was not provided
        '''
        self.label = df.get(df.keys()[0])
        self.KU = df.get(df.keys()[1])
        self.topics = df.get(df.keys()[2])
        self.descriptions = df.get(df.keys()[3])

    def clean_text(self, text): 
        #special_chars = r'[\_\-]'
        #cleaned_text = re.sub(special_chars,' ', text)        # replace - or _ with space 
        #cleaned_text = re.sub(r'[^a-zA-Z0-9 ]', '', text)     # remove any non-alphanumeric /space character
        return text.lower() 
        
    def index(self, idx=None): 
        if idx == None:                                        # choose random index if index not provided
            idx = np.random.randint(0,len(self.label))
         
        description = self.clean_text(self.descriptions[idx])  # clean topics & descriptions 
        topic = self.clean_text(self.topics[idx])              # clean topics & descriptions 
        if (topic == ""):                                      # return knowledge unit if there is no topic
            topic = self.clean_text(self.KU[idx])
        
        if isinstance(self.label[idx],int):                    # label is either an int or a messy string
            label = str(self.label[idx])
        else: 
            label = self.clean_text(self.label[idx])
        label_nospace = label.replace(' ','')
        return topic, description, label_nospace
        
df = pd.read_excel('data/class1_8.ods', engine='odf')
read = read_topic(df)
topic, description, label = read.index(0)
print(description)
# empty lines possible 

this topic covers basic concepts in cryptography to  build the base for other sections in the knowledge  unit. this topic includes: ● encryption/decryption, sender authentication,  data integrity, non-repudiation, ● attack classification (ciphertext-only, known  plaintext, chosen plaintext, chosen ciphertext),  ● secret key (symmetric), cryptography and publickey (asymmetric) cryptography,  ● information-theoretic security (one-time pad,  shannon theorem), and • computational security 


In [3]:
def submit_message_LLM(model, messages): 
    text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(device)
    
    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=512
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    
    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

In [4]:
# Load Qwen2-7B-Instruct model. 
from transformers import AutoModelForCausalLM, AutoTokenizer
API_KEY="<YOUR-APKI-KEY-HERE>"
model_path="Qwen/Qwen2.5-7B-Instruct"
device = "cuda" # the device to load the model onto

model2 = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_path)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [5]:
import json
from tqdm import tqdm
def topic_prompt(topic, description):
        """
        Formats a topic and description into a list of message dictionaries for the pipeline.
        """
        #options_str = ', '.join([f"{key}) {value}" for key, value in answers.items()])
        instructions = (
            "You are a helpful AI assistant.\n"
            "Instructions:\n"
            "a. Carefully read the topic and description .\n"
            "b. Give a list of all subtopics in the description .\n"
            "c. Do NOT include any explanation or additional text in the response.\n"
            #"d. Always return the answer in this XML format: '<xml>answer</xml>'. "
            #"For example, if the correct answer is D, then return <xml>D</xml>.\n\n"
        )
    
        messages = [
            {"role": "system", "content": instructions},
            {"role": "user", "content": f"#topic : {topic}\n #description: {description}"}
        ]
        return messages
    
# load topic, description and label of certain index using read_topic class 
#idx = 7
read = read_topic(df)
topic, description, label = read.index()
print('description: ',description)
print(label)
# create prompt based on topic + description
#message = topic_prompt(topic, description)
#print('message: ',message)
# put the formatted prompt through the model: 
#response = submit_message_LLM(model2, message)
#print('response: ',response)

description:  this knowledge unit covers efforts to enhance the  security of the origin and traceability of sourced system  components, such as externally produced hardware or  software. 
7


In [6]:
# re-format output 
def clean_text(text): 
    special_chars = r'[\_/-]'
    cleaned_text = re.sub(special_chars,' ', text)                # replace - or _ or / with space 
    cleaned_text = re.sub(r'[^a-zA-Z0-9 -/()]', '', cleaned_text) # remove any non-alphanumeric /space character
    return cleaned_text.lower()                                   # convert to lowercase 


In [7]:
read = read_topic(df)                                   # initialize read_topic class 
descriptions = [] 
topics = []
labels = []
for idx in range(len(read.label)):
    print('processing topic: ', idx)
    topic, description, label = read.index(idx)
    #print('description: ',description)
    if (topic != ""):                                   # check if not empty! 
        message = topic_prompt(topic, description)      # create prompt based on topic + description
        response = submit_message_LLM(model2, message)  # put the formatted prompt through the LLM 
        #print('response: ',response)
        subtopics = response.split("\n")                # split response into subtopics
        for sub in subtopics: 
            sub = clean_text(sub[2:])
            topics.append(sub)                      # add subtopics and the original label to lists 
            labels.append(label)
            descriptions.append(description)

processing topic:  0


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


processing topic:  1
processing topic:  2
processing topic:  3
processing topic:  4
processing topic:  5
processing topic:  6
processing topic:  7
processing topic:  8
processing topic:  9
processing topic:  10
processing topic:  11
processing topic:  12
processing topic:  13
processing topic:  14
processing topic:  15
processing topic:  16
processing topic:  17
processing topic:  18
processing topic:  19
processing topic:  20
processing topic:  21
processing topic:  22
processing topic:  23
processing topic:  24
processing topic:  25
processing topic:  26
processing topic:  27
processing topic:  28
processing topic:  29
processing topic:  30
processing topic:  31
processing topic:  32
processing topic:  33
processing topic:  34
processing topic:  35
processing topic:  36
processing topic:  37
processing topic:  38
processing topic:  39
processing topic:  40
processing topic:  41
processing topic:  42
processing topic:  43
processing topic:  44
processing topic:  45
processing topic:  

In [8]:
import pandas as pd
#print(topics)
#print(labels)
# turn KDs into a dataframe 
df_KDs = {'Statement Description': descriptions,'topics': topics, 'label':labels}
df_KDs = pd.DataFrame(df_KDs)
print(df_KDs)

                                  Statement Description  \
0     this topic covers basic concepts in cryptograp...   
1     this topic covers basic concepts in cryptograp...   
2     this topic covers basic concepts in cryptograp...   
3     this topic covers basic concepts in cryptograp...   
4     this topic covers basic concepts in cryptograp...   
...                                                 ...   
1645  this topic covers the role of corporations in ...   
1646  this topic includes: ●privacy rights and threa...   
1647  this topic includes: ●privacy rights and threa...   
1648  this topic includes: ●privacy rights and threa...   
1649  this topic includes: ●privacy rights and threa...   

                                                 topics      label  
0                                 encryption decryption          1  
1                                 sender authentication          1  
2                                        data integrity          1  
3              

In [9]:
# 2. Format all labels as one-hot encodings. 
# ideally import the function down below. 
def df_to_onehot(df, column=' Knowledge areas (CSEC2017)'): 
    # find all unique labels
    all_labels = set(label.replace('\xa0','') for row in df[column] for label in row.split(','))
    all_labels = list(all_labels)
    #all_labels = [labels.replace('\xa0','') for labels in all_labels]
    #all_labels = [str(i) for i in range(1,9)]
    print(all_labels)
    all_labels.sort()
    print(all_labels)
    # initialize one-hot encoding coluns with 0. 
    for label in all_labels[1:]: 
        df[label] = 0 
    # set 1 for all indicated labels in the specified column. 
    for index, row in df.iterrows(): 
        labels = row[column].split(',')
        for label in labels: 
            label = label.replace('\xa0','')
            label = label.replace(' ','')
            label = label.strip()
            if label not in all_labels: 
                print(label)
            df.at[index,label] = 1
df_to_onehot(df_KDs, column='label')
print(df_KDs)

['', '2', '6', '5', '1', '3', '8', '7', '4']
['', '1', '2', '3', '4', '5', '6', '7', '8']
                                  Statement Description  \
0     this topic covers basic concepts in cryptograp...   
1     this topic covers basic concepts in cryptograp...   
2     this topic covers basic concepts in cryptograp...   
3     this topic covers basic concepts in cryptograp...   
4     this topic covers basic concepts in cryptograp...   
...                                                 ...   
1645  this topic covers the role of corporations in ...   
1646  this topic includes: ●privacy rights and threa...   
1647  this topic includes: ●privacy rights and threa...   
1648  this topic includes: ●privacy rights and threa...   
1649  this topic includes: ●privacy rights and threa...   

                                                 topics      label  1  2  3  \
0                                 encryption decryption          1  1  0  0   
1                                 sender au

In [10]:
# save dataset
df_KDs.to_csv("data/train_CSEC2017.csv")